# Run fgsea on deseq2 results

## 2025-10-23
### Palak Genge, High Resolution Translational Immunology, Allen Institute for Immunology
#### Objective: Run fgsea analysis on deseq2 outputs

In [47]:
# load all required packages
suppressPackageStartupMessages({
  library(data.table)  
  library(dplyr)       
  library(tidyr)       
  library(stringr)     
  library(ggplot2)     
  library(ggrepel)     
  library(grid)        
  library(DESeq2)      
  library(qvalue)      
  library(parallel)    
  library(tidyverse)
  library(purrr)
  library(fgsea)
})

In [2]:
# read in the gmt reference files
gmt_dir <- "../../../data/gmt"
gmt_files <- list.files(gmt_dir, pattern = "*.gmt$", full.names = TRUE)

# read each gmt and combine into a single list
pathways_list <- lapply(gmt_files, gmtPathways)
pathways <- do.call(c, pathways_list)  # merge all pathways

# list all deg csvs for each tumor cluster
deg_files <- list.files("../../../data/rna/plasma/outputs/DEGs", pattern = "*.csv", full.names = TRUE)

# loop over each tumor cluster
for (deg_file in deg_files) {
  
  # extract cluster name
  cluster_name <- gsub("DEGs_|_vs_Healthy.csv", "", basename(deg_file))
  
  # read DEGs with gene names as row names
  degs <- read.csv(deg_file, row.names = 1)
  
  # filter for significant DEGs only
  sig_degs <- degs[!is.na(degs$padj) & degs$padj < 0.05, ]
  
  if (nrow(sig_degs) < 15) {
    warning("Not enough significant genes for FGSEA for cluster ", cluster_name)
    next
  }
  
  # prepare named vector of log2FoldChange
  ranks <- setNames(sig_degs$log2FoldChange, rownames(sig_degs))
  
  # run fgsea using fgseaMultilevel
  fgseaRes <- fgsea(
    pathways = pathways,
    stats = ranks,
    minSize = 15,
    maxSize = 500
  )
  
  # add direction of regulation based on NES
  fgseaRes <- fgseaRes %>%
    mutate(direction = ifelse(NES > 0, "up", "down")) %>%
    arrange(desc(NES))
  
  # convert list column for writing out to csv
  fgseaRes$leadingEdge <- sapply(fgseaRes$leadingEdge, function(x) paste(x, collapse = ";"))
  
  # save as csv
  out_file <- paste0("../../../data/rna/plasma/outputs/FGSEA/FGSEA_", cluster_name, "_vs_Healthy_signifDEGs.csv")
  dir.create(dirname(out_file), showWarnings = FALSE, recursive = TRUE)
  write.csv(fgseaRes, out_file, row.names = FALSE)
  # output so know it's done
  message("✅ FGSEA done for ", cluster_name, " → ", out_file)
}

Warning message:
“package ‘data.table’ was built under R version 4.4.3”

Attaching package: ‘data.table’


The following object is masked from ‘package:SummarizedExperiment’:

    shift


The following object is masked from ‘package:GenomicRanges’:

    shift


The following object is masked from ‘package:IRanges’:

    shift


The following objects are masked from ‘package:S4Vectors’:

    first, second


Warning message:
“package ‘dplyr’ was built under R version 4.4.3”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following object is masked from ‘package:Biobase’:

    combine


The following object is masked from ‘package:matrixStats’:

    count


The following objects are masked from ‘package:GenomicRanges’:

    intersect, setdiff, union


The following object is masked from ‘package:GenomeInfoDb’:

    intersect


The following objects are masked from ‘package:IRanges’:

    collapse, desc, intersect, s